# 9주차: 검색 결합 생성(RAG)

- 공개 원문 → 문단 색인 → 검색 → 재순위 → 근거 문장 → 짧은 생성의 제공 골격
- 수정할 곳: 검색 함수 선택 한 줄. 기본 상태로 전체 실행 가능
- CPU 실행, GPU·외부 유료 호출 불필요. 모델 다운로드에 네트워크와 저장 공간 필요
- 데이터: 위키백과 「제주특별자치도」 고정 판 42464087, CC BY-SA 4.0
- 원문 및 저자 목록: https://ko.wikipedia.org/w/index.php?oldid=42464087
- 원 논문 RAG의 공동 학습을 재현하는 실험이 아닌 검색-생성 기준선


In [ ]:
# 필요한 라이브러리 설치. 새 런타임에서 한 번 실행
%pip -q install scikit-learn transformers torch


## 1. 공개 원문 수집

- 7주차와 같은 고정 판 사용
- 원문은 실행 때 다운로드. 네트워크 실패 시 가짜 자료로 대체하지 않고 중단
- 2026-10-11 작성 환경에서 해당 API HTTP 200 확인. 다른 환경의 성공 보장과 구별


In [ ]:
import html, json, re, time, urllib.parse, urllib.request
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

OLDID = 42464087
SOURCE = f"https://ko.wikipedia.org/w/index.php?oldid={OLDID}"
api = "https://ko.wikipedia.org/w/api.php?" + urllib.parse.urlencode({
    "action": "parse", "oldid": OLDID, "prop": "text", "format": "json", "formatversion": 2})
request = urllib.request.Request(api, headers={"User-Agent": "deepnlp-course/1.0 (yj.lee@chu.ac.kr)"})
with urllib.request.urlopen(request, timeout=60) as response:
    parsed = json.load(response)["parse"]
assert parsed["revid"] == OLDID, "수집 판 번호 불일치"

def clean(fragment):
    fragment = re.sub(r"<sup\b[^>]*>.*?</sup>", "", fragment, flags=re.S)
    fragment = re.sub(r"<[^>]+>", " ", fragment)
    return re.sub(r"\s+", " ", html.unescape(fragment)).strip()

paragraphs = [clean(p) for p in re.findall(r"<p\b[^>]*>(.*?)</p>", parsed["text"], flags=re.S)]
paragraphs = [p for p in paragraphs if len(p) >= 40]
assert len(paragraphs) >= 5, "원문 문단 부족: API 응답과 전처리 확인"
documents = [{"id": f"D{i:03d}", "text": p, "source": SOURCE} for i, p in enumerate(paragraphs)]
print("출처:", SOURCE)
print("제목:", parsed["title"], "판:", parsed["revid"], "문단:", len(documents))
print(documents[0]["id"], documents[0]["text"][:180])


## 2. 같은 문서로 두 색인 생성

- 단어 TF-IDF: 공백과 단어 패턴에 따른 특징
- 문자 TF-IDF: 길이 2~4의 문자 부분 문자열
- 두 표현은 학습된 의미 벡터가 아닌 희소 표현
- 점수는 L2 정규화한 벡터의 내적. 서로 다른 색인의 점수를 확률처럼 비교 금지


In [ ]:
word_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
char_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
texts = [d["text"] for d in documents]
word_index = word_vectorizer.fit_transform(texts)
char_index = char_vectorizer.fit_transform(texts)
print("단어 색인:", word_index.shape, "문자 색인:", char_index.shape)

# 두 함수는 같은 입출력 계약: 질문과 k를 받아 문단 번호·검색 점수 반환
def search_index(query, vectorizer, index, k):
    scores = (index @ vectorizer.transform([query]).T).toarray().ravel()
    order = np.argsort(-scores, kind="stable")[:k]
    return [(int(i), float(scores[i])) for i in order if scores[i] > 0]

def retrieve_word(query, k=5):
    return search_index(query, word_vectorizer, word_index, k)

def retrieve_char(query, k=5):
    return search_index(query, char_vectorizer, char_index, k)


### TF-IDF와 토큰 특징 계산 확인

- A/B/C는 공식 계산용 짧은 문자열. 제주 공개 원문 데이터가 아님
- 강의의 IDF와 코사인 수치를 라이브러리로 검산
- 실제 질의의 단어 특징 중 색인 어휘에 있는 것 확인, 부분 문자열의 의미 한계 토론


In [ ]:
toy = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
toy_index = toy.fit_transform(["제주 기후 기후", "제주 관광", "제주"])
toy_query = toy.transform(["제주 기후"])
print("계산 예제 특징 순서:", toy.get_feature_names_out().tolist())
print("IDF:", dict(zip(toy.get_feature_names_out(), np.round(toy.idf_, 4))))
print("A/B/C 정규화 행렬:\n", np.round(toy_index.toarray(), 4))
print("질의 코사인 A/B/C:", np.round((toy_index @ toy_query.T).toarray().ravel(), 4))
word_analyzer = word_vectorizer.build_analyzer()
features = word_vectorizer.vocabulary_
for query in ["한라산", "한라산의 높이", "제주도의 기후"]:
    terms = word_analyzer(query)
    print("질의:", query, "단어 특징:", terms,
          "색인 어휘와 겹침:", [t for t in terms if t in features])
print("문자 2-gram 예:", ["한라산의"[i:i+2] for i in range(len("한라산의") - 1)])


## 3. 재순위와 근거 선택 골격

- 재순위: 질의 토큰의 문단 내 포함 비율로 정렬하는 간단한 기준선
- 학습된 cross-encoder와 구별. 후보 밖 문서는 추가하지 않는 구조
- 근거 선택: 상위 문단에서 질의 토큰이 겹치는 문장 우선 추출
- 추출 문장은 생성 모델에 전달할 문맥, 생성 답변과 구별


In [ ]:
def query_terms(query):
    return set(re.findall(r"\w+", query))

def overlap(query, text):
    terms = query_terms(query)
    return sum(term in text for term in terms) / max(1, len(terms))

def rerank(query, candidates):
    return sorted(candidates, key=lambda item: (-overlap(query, documents[item[0]]["text"]), -item[1], item[0]))

def evidence_sentence(query, text):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    return max(sentences, key=lambda sentence: overlap(query, sentence)) if sentences else text

def run_pipeline(query, retrieve, k=5):
    candidates = retrieve(query, k)
    ranked = rerank(query, candidates)
    if not ranked:
        return {"query": query, "candidate_ids": [], "ranked_ids": [], "evidence_id": None,
                "evidence": "", "retrieval_score": 0.0}
    top, score = ranked[0]
    return {"query": query,
            "candidate_ids": [documents[i]["id"] for i, _ in candidates],
            "ranked_ids": [documents[i]["id"] for i, _ in ranked],
            "evidence_id": documents[top]["id"],
            "evidence": evidence_sentence(query, documents[top]["text"]),
            "retrieval_score": score}


## 4. 단어 검색 기준선

- 지명 질문, 조사 포함 질문, 원문에 최신 답이 있는지 확인할 질문 구분
- 마지막 질문의 답을 원문에 없는 최신 사실로 채우지 않는 원칙
- 검색 결과가 비어 있지 않다는 것과 근거가 맞다는 것의 차이 확인


In [ ]:
queries = ["한라산", "한라산의 높이", "제주도의 기후", "제주 관광지 오늘 운영 시간"]
baseline = [run_pipeline(q, retrieve_word) for q in queries]
for result in baseline:
    print("\n질문:", result["query"])
    print("검색 후보:", result["candidate_ids"], "재순위:", result["ranked_ids"])
    print("선택 근거:", result["evidence_id"], result["evidence"][:300])


## 5. 검색 단계 하나 교체

- 아래 한 줄만 `retrieve = retrieve_char`로 변경한 뒤 이 셀부터 다시 실행
- 문서, 질문, k, 재순위, 생성 모델은 동일하게 유지
- 기본값은 단어 검색. 수정하지 않아도 나머지 셀 실행 가능


In [ ]:
# TODO: 검색 함수 선택 한 곳만 변경 (retrieve_word → retrieve_char)
retrieve = retrieve_word


In [ ]:
changed = [run_pipeline(q, retrieve) for q in queries]
for before, after in zip(baseline, changed):
    print("\n질문:", after["query"])
    print("기준선 후보:", before["candidate_ids"])
    print("교체 후보:", after["candidate_ids"])
    print("기준선 근거:", before["evidence_id"], "교체 근거:", after["evidence_id"])
    print("교체 근거 문장:", after["evidence"][:300])


## 6. 같은 모델로 짧은 답변 생성

- 4~7주차의 Qwen3-0.6B-Base를 CPU에서 사용, 검색기 학습·미세조정 없음
- 근거가 있는 기후 질의의 전후 생성으로 기본 상태에서도 실제 모델 출력 확인
- 높이 질의의 실패도 전후 생성 비교에 보존. 후보가 없으면 그 질의는 생성 생략
- 각 답변 최대 32토큰, 총 최대 4회 생성. 전달한 문장 최대 600자, 입력 최대 768토큰
- Base 모델은 지시 수행이 불완전할 수 있음. 틀린 출력과 미완성 답도 원문 그대로 기록
- 답의 정확성·근거 충실성은 아래 수동 판정 필요


In [ ]:
import torch, transformers, sklearn, platform, hashlib
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.set_num_threads(2)
torch.manual_seed(0)
MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
MODEL_REVISION = "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
GENERATION_SETTINGS = {"max_new_tokens": 32, "do_sample": False, "use_cache": True}
execution_conditions = {
    "model": MODEL_NAME, "model_revision": MODEL_REVISION,
    "tokenizer_revision": MODEL_REVISION, "device": "cpu", "dtype": "float32",
    "torch_threads": torch.get_num_threads(), "seed": 0,
    "python": platform.python_version(), "machine": platform.machine(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "scikit_learn": sklearn.__version__, "numpy": np.__version__,
    "context_char_limit": 600, "prompt_token_limit": 768,
    "truncation_side": "right", "generation": GENERATION_SETTINGS,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
print("실행 조건:", json.dumps(execution_conditions, ensure_ascii=False))
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, torch_dtype=torch.float32).to("cpu").eval()

tokenizer.truncation_side = "right"

def generate_answer(result):
    context = result["evidence"][:600]
    if not context:
        return {"status": "생성 생략: 검색 근거 없음", "answer": "검색 근거 없음",
                "context": "", "actual_input_text": None, "seconds": None,
                "input_tokens_before_truncation": None, "actual_input_tokens": None,
                "output_tokens": None, "prompt_truncated": None, "context_char_truncated": None,
                "prompt_sha256": None, "actual_input_ids_sha256": None}
    prompt = ("제공된 근거만 사용해 질문에 짧게 답하세요. 근거에 답이 없으면 '근거 부족'으로 답하세요.\n"
              f"근거 [{result['evidence_id']}]: {context}\n질문: {result['query']}\n답변:")
    full_ids = tokenizer(prompt, add_special_tokens=True).input_ids
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    actual_ids = inputs["input_ids"][0].tolist()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, **GENERATION_SETTINGS,
                                pad_token_id=tokenizer.eos_token_id)
    elapsed = time.perf_counter() - start
    new_ids = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_ids, skip_special_tokens=True)
    input_hash = hashlib.sha256(json.dumps(actual_ids, separators=(",", ":")).encode()).hexdigest()
    observation = {"status": "생성 완료", "answer": answer, "context": context,
                   "actual_input_text": tokenizer.decode(actual_ids, skip_special_tokens=False),
                   "seconds": elapsed,
                   "input_tokens_before_truncation": len(full_ids), "actual_input_tokens": len(actual_ids),
                   "output_tokens": int(new_ids.numel()), "prompt_truncated": len(full_ids) > len(actual_ids),
                   "context_char_truncated": len(result["evidence"]) > len(context),
                   "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                   "actual_input_ids_sha256": input_hash}
    assert observation["actual_input_tokens"] <= 768 and observation["output_tokens"] <= 32
    print("입력 추적:", {key:value for key,value in observation.items()
                            if key not in ("answer", "context", "actual_input_text")})
    return observation

# 기후 질의에는 단어 검색 근거가 있어 기본 상태에도 실제 생성 실행
climate_before = generate_answer(baseline[2])
climate_after = generate_answer(changed[2])
print("기후 기준선 답변:", climate_before["answer"])
print("기후 교체 답변:", climate_after["answer"])

# 높이 질의의 후보 누락·무관한 근거 실패 보존
answer_before = generate_answer(baseline[1])
answer_after = generate_answer(changed[1])
print("기준선 답변:", answer_before["answer"])
print("교체 답변:", answer_after["answer"])
print("실행 시간(환경 의존):", answer_before["seconds"], answer_after["seconds"])


## 7. 예비 지표와 수동 검증

- 아래 `proxy_hit`는 지정 문자열 포함 문단의 후보 등장 여부. 정확성 지표로 대체 금지
- 문단의 내용이 질문에 직접 답하는지는 사람이 확인
- `human_labels`의 미판정 상태를 그대로 저장. 미판정을 실패나 성공으로 자동 처리하지 않는 원칙
- 원문에 답이 없는 질문은 근거 없음으로 기록, 최신 운영 정보 생성 금지


## 7-1. 공개 표 어댑터와 수집 상태

- 교수 제공 완성 어댑터. 학생이 CSV 파서·색인·재순위를 새로 작성하지 않는 구성
- 교수 준비물: `week09-public.csv` 또는 `week09-public.json`과 `week09-public-source.json`
- CSV는 UTF-8, JSON은 행 객체의 배열. 원본 전체는 결과 JSON에 저장하지 않음
- 출처 설정 계약: `source`(지정 두 포털의 자료 페이지), `license`(확인한 이용 조건), `fetched_at_utc`, `file_sha256`, `text_columns`(확인한 공개 텍스트 필드 이름 배열), `id_column`(필드 이름 또는 null)
- 필드 이름은 확인된 실제 파일에 맞춰 교수자가 설정. 개인정보·연락처 필드 미선택, 기업 자료 사용 금지
- 실습 규모: 앞 50행만 색인, 전체 자료 품질로 일반화 금지
- 기본은 공개 포털 파일 미제공 상태. 행 수·검색 성공·평가가 미확인이면 null로 기록, 0건이나 성공으로 바꾸지 않는 규칙
- 2026-10-11: 관광 포털 `regSn=20` 공개 페이지의 CSV 형식 메타데이터는 검색으로 확인했으나 HTTPS 다운로드 검증·실제 파일·컬럼·이용 조건 미확인. 데이터허브도 실제 파일 미확인
- 파일 없이 기본 실행을 마쳐도 포털 자료 색인 과제 완료라는 의미 없음


In [ ]:
import csv, hashlib, io
from datetime import datetime, timezone

PORTAL_SOURCE_DOMAINS = {"jejudatahub.net", "www.jejudatahub.net", "data.ijto.or.kr"}

def load_public_table():
    metadata_path = Path("week09-public-source.json")
    candidates = [path for path in (Path("week09-public.csv"), Path("week09-public.json")) if path.is_file()]
    state = {"status": "미수집", "reason": "교수가 확인한 공개 표와 출처 설정 파일 미제공",
             "source": None, "license": None, "fetched_at_utc": None, "file_sha256": None,
             "row_count": None, "indexed_rows": None, "columns": None, "queries": None}
    if not candidates or not metadata_path.is_file():
        return state, None
    try:
        if len(candidates) != 1:
            raise ValueError("CSV 또는 JSON 원본 중 하나만 준비")
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        required = ("source", "license", "fetched_at_utc", "file_sha256", "text_columns")
        if any(not metadata.get(key) for key in required):
            raise ValueError("출처·이용 조건·수집 시각·hash·텍스트 컬럼 미확인")
        source = urllib.parse.urlparse(metadata["source"])
        if source.scheme != "https" or source.hostname not in PORTAL_SOURCE_DOMAINS:
            raise ValueError("지정 두 공개 포털의 HTTPS 자료 페이지 출처 필요")
        parsed_time = datetime.fromisoformat(metadata["fetched_at_utc"].replace("Z", "+00:00"))
        if parsed_time.tzinfo is None:
            raise ValueError("수집 시각에 시간대 필요")
        columns = metadata["text_columns"]
        if not isinstance(columns, list) or not columns or any(not isinstance(x, str) or not x for x in columns):
            raise ValueError("실제로 확인한 공개 텍스트 필드 배열 필요")
        data_path = candidates[0]
        if data_path.stat().st_size > 2_000_000:
            raise ValueError("무료 CPU 실습은 2MB 이내 공개 표 사용")
        raw = data_path.read_bytes()
        digest = hashlib.sha256(raw).hexdigest()
        if digest != metadata["file_sha256"]:
            raise ValueError("확인한 원본 파일 hash와 불일치")
        if data_path.suffix == ".csv":
            rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))))
        else:
            rows = json.loads(raw.decode("utf-8-sig"))
        if not isinstance(rows, list) or not rows or any(not isinstance(row, dict) for row in rows):
            raise ValueError("비어 있지 않은 행 객체 배열 필요")
        id_column = metadata.get("id_column")
        expected_columns = columns + ([id_column] if id_column else [])
        if any(any(key not in row for key in expected_columns) for row in rows):
            raise ValueError("확인한 컬럼과 실제 파일의 필드 불일치")
        docs = []
        for i, row in enumerate(rows[:50]):
            values = [str(row[key]).strip() for key in columns if row[key] is not None]
            text = " | ".join(values)
            if not text.strip():
                raise ValueError("선택한 공개 텍스트 필드가 비어 있는 행 존재")
            identifier = str(row[id_column]) if id_column else f"P{i:03d}"
            docs.append({"id": identifier, "text": text, "source": metadata["source"]})
        if len({doc["id"] for doc in docs}) != len(docs):
            raise ValueError("색인 범위의 중복 ID, 출처 설정 확인 필요")
        state.update(status="수집·어댑터 확인", reason="교수 제공 파일의 출처 설정·hash·컬럼 대조",
                     source=metadata["source"], license=metadata["license"],
                     fetched_at_utc=metadata["fetched_at_utc"], file_sha256=digest,
                     row_count=len(rows), indexed_rows=len(docs), columns=columns)
        return state, docs
    except (OSError, UnicodeError, ValueError, KeyError, TypeError, AttributeError) as error:
        state.update(status="미확인", reason=str(error))
        return state, None

def compare_public_table(docs):
    vectorizers = {"retrieve_word": TfidfVectorizer(token_pattern=r"(?u)\b\w+\b"),
                   "retrieve_char": TfidfVectorizer(analyzer="char", ngram_range=(2, 4))}
    results = []
    for method, vectorizer in vectorizers.items():
        index = vectorizer.fit_transform([doc["text"] for doc in docs])
        for query in queries:
            candidates = search_index(query, vectorizer, index, 5)
            ranked = sorted(candidates, key=lambda item: (-overlap(query, docs[item[0]]["text"]), -item[1], item[0]))
            result = {"method": method, "query": query, "candidate_ids": [docs[i]["id"] for i,_ in candidates],
                      "ranked_ids": [docs[i]["id"] for i,_ in ranked], "gold_ids": None,
                      "answerable": None, "failure_type": "미판정", "generation": None}
            if ranked:
                i, score = ranked[0]
                evidence = evidence_sentence(query, docs[i]["text"])
                result.update(evidence_id=docs[i]["id"], evidence=evidence, retrieval_score=score)
            else:
                result.update(evidence_id=None, evidence="", retrieval_score=None)
            results.append(result)
    return results

portal_observation, portal_documents = load_public_table()
if portal_documents is not None:
    try:
        portal_observation["queries"] = compare_public_table(portal_documents)
    except ValueError as error:
        portal_observation.update(status="색인 미확인", reason=str(error), queries=None)
print("공개 포털 자료 상태:", json.dumps(portal_observation, ensure_ascii=False, indent=2))
print("포털 생성·정답 판정:", "미측정. 위키백과 생성 실험과 별도 비교")


In [ ]:
anchors = {"한라산": "한라산", "한라산의 높이": "한라산", "제주도의 기후": "기후"}
def proxy_hit(result):
    anchor = anchors.get(result["query"])
    if anchor is None:
        return None
    relevant_ids = {d["id"] for d in documents if anchor in d["text"]}
    if not relevant_ids:
        return None
    return bool(relevant_ids.intersection(result["candidate_ids"]))

for before, after in zip(baseline, changed):
    print(after["query"], "문자열 기준 후보 포함:", proxy_hit(before), "→", proxy_hit(after))

human_labels = [{"query": q, "gold_ids": [], "answerable": None,
                 "failure_type": "미판정", "reason": "원문과 전달된 근거를 직접 읽은 뒤 판정"} for q in queries]
report = {"execution_conditions": execution_conditions, "portal_data": portal_observation, "source": SOURCE, "revision": OLDID, "license": "CC BY-SA 4.0",
          "method_before": "retrieve_word", "method_after": retrieve.__name__, "k": 5,
          "baseline": baseline, "changed": changed,
          "generated_before": answer_before, "generated_after": answer_after,
          "generated_climate_before": climate_before, "generated_climate_after": climate_after,
          "proxy_hit_before": [proxy_hit(r) for r in baseline],
          "proxy_hit_after": [proxy_hit(r) for r in changed], "human_labels": human_labels}
output_path = Path("week09-results.json")
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", output_path)


## 실패 판정 안내

- 수집 범위: 원문에 답이 없음
- 검색 실패: 정답 근거가 원문에 있으나 후보에 없음
- 재순위 실패: 후보에 정답이 있으나 다른 문단이 먼저 선택
- 문맥 절단: 선택 문단의 필요한 부분이 600자 제한으로 잘림
- 근거 사용 실패: 전달된 문맥으로 답할 수 있으나 생성 답변이 틀림
- 원문에 없는 질문에서 답변 유보 여부 확인

## 과제 3에 연결

- 제주데이터허브·제주 관광 빅데이터 서비스 플랫폼 공개 데이터 실물과 이용 조건 확인 필요
- 이 위키백과 골격 실행만으로 두 포털 데이터 색인 과제 전체를 완료한 것으로 보고하지 않는 원칙
- 아래 제공된 CSV/JSON 어댑터로 같은 문서 계약(id, text, source) 사용. 교수자가 확인한 원본 파일과 출처·필드 설정 파일을 준비하며 학생의 추가 코드 작성 없음
- 원본·설정 파일이 없으면 포털 실습은 미수집, 컬럼이나 이용 조건이 맞지 않으면 미확인으로 저장. 위키백과 골격 실행과 별도 상태
- 제출물에 데이터 출처·수집 시각·검색 방법·실패 유형 기록, 데이터 원본과 개인정보 미포함

## 읽기

- RAG: https://arxiv.org/abs/2005.11401
- DPR: https://arxiv.org/abs/2004.04906
- Lost in the Middle: https://arxiv.org/abs/2307.03172
- GraphRAG: https://arxiv.org/abs/2404.16130


## 돌아보기: 확인 질문

1. 제공된 작은 어휘표에서 IDF·정규화·cosine 계산이 출력과 일치하는지 확인
2. 높이 질문의 후보 누락과 무관한 문장 선택을 서로 다른 실패로 판정
3. 기후 질의에서 검색 후보, 실제 입력 문맥, 생성 답을 따로 대조
4. proxy_hit가 True여도 정답 근거 확보를 뜻하지 않는 사례 설명

## 제출 안내

- 코드 구현이나 자료 수집을 새로 요구하는 과제가 아닌 제공된 실습의 관찰 기록
- 별도 제출 공지가 있을 때 실행 결과 JSON과 질문별 관찰·판정 이유를 함께 제출
- 저장된 결과에서 미확인·미판정은 그대로 표시, 모델 학습 성과로 보고하지 않는 원칙
- 학생 폴더에는 학번만 사용, 이름·연락처·데이터 원본·API 키 제외
